## LLM-as-a-Judge Pipeline

**What this notebook does:** Runs one or more LLM judge experiments over a dataset of multi-agent traces and evaluates the predictions against annotations. Reports accuracy, precision, recall, F1, and Cohen's kappa.

**How to use this template:**
1. Copy this folder to a new experiment directory
2. Edit `config.yaml` to define your experiments (one or more)
3. Run this notebook top to bottom

**Inputs:**
- `config.yaml` in the same folder: defines a list of experiments, each with model, backend, dataset path, shots, etc.
- Dataset: set `dataset_path` in config to either `MAD_human_labelled_dataset.json` or `MAD_full_dataset.json`

**Outputs (saved to `saved_results/`):**
- `predictions.csv`  per-trace predictions with token counts and cost for all experiments
- `metrics_per_mode.csv`  per-failure-mode breakdown for all experiments
- `summary.csv`  one row per experiment with overall metrics (accuracy, F1, kappa, 95% CIs)
- `checkpoints/<name>.pkl`  checkpoint saved after every trace (allows resuming)

**Run order:** Top to bottom.

### 1. Setup

Loads `config.yaml`, initialises the `LLMJudge`, and loads the dataset.

In [1]:
import numpy as np
import pandas as pd
import re
import pickle
import json
import os
import sys
from sklearn.metrics import f1_score, precision_score, recall_score, cohen_kappa_score, accuracy_score

# Find repo root regardless of working directory
_root = os.path.abspath(os.getcwd())
while not os.path.isdir(os.path.join(_root, "LLM_models_interface")):
    _root = os.path.dirname(_root)
sys.path.insert(0, _root)

from LLM_models_interface.llm_interface import LLMJudge, load_configs, load_dataset

configs = load_configs("config.yaml")

### 2. Run the judge

Iterates over every trace and calls `judge.judge_trace()`. Results are checkpointed to `saved_results/` after each trace, if the run is interrupted, completed results are not lost.

In [ ]:
official_modes = ['1.1','1.2','1.3','1.4','1.5',
                  '2.1','2.2','2.3','2.4','2.5','2.6',
                  '3.1','3.2','3.3']

def bootstrap_ci(y_true, y_pred, metric_fn, n=1000, ci=0.95):
    rng = np.random.default_rng(42)
    scores = []
    idx = np.arange(len(y_true))
    yt, yp = np.array(y_true), np.array(y_pred)
    for _ in range(n):
        s = rng.choice(idx, size=len(idx), replace=True)
        scores.append(metric_fn(yt[s], yp[s]))
    lo = np.percentile(scores, (1 - ci) / 2 * 100)
    hi = np.percentile(scores, (1 + ci) / 2 * 100)
    return lo, hi

def trace_id_of(record):
    is_full = isinstance(record['trace'], dict)
    return f"{record['trace']['key']}_{record['trace_id']}" if is_full else record['trace_id']

def build_gt_lookup(dataset, modes):
    """Returns {trace_id: {mode: 0|1}} for every record in dataset."""
    lookup = {}
    for record in dataset:
        tid = trace_id_of(record)
        gt = {}
        if "mast_annotation" in record:
            for code in modes:
                gt[code] = record["mast_annotation"].get(code, 0)
        else:
            for ann in record.get("annotations", []):
                m = re.match(r"(\d+\.\d+)", ann["failure mode"])
                if not m:
                    continue
                code = m.group(1)
                if code not in modes:
                    continue
                votes = [ann["annotator_1"], ann["annotator_2"], ann["annotator_3"]]
                gt[code] = 1 if sum(votes) >= 2 else 0
        lookup[tid] = gt
    return lookup

summary = []
all_metrics_per_mode = []
all_predictions = []

for cfg in configs:
    print(f"\n{'='*60}")
    print(f"Experiment: {cfg.name} — {cfg.model}")
    print(f"{'='*60}")

    judge = LLMJudge(cfg)
    traces = load_dataset(cfg)
    if traces and "round" in traces[0]:
        traces = [t for t in traces if t.get("round") == "Round 3"]

    os.makedirs('saved_results/checkpoints', exist_ok=True)
    checkpoint_path = f'saved_results/checkpoints/{cfg.name}.pkl'

    # Resume from checkpoint — skips already-completed traces
    if os.path.exists(checkpoint_path):
        with open(checkpoint_path, 'rb') as f:
            results = pickle.load(f)
        completed_ids = {r.trace_id for r in results}
        traces = [t for t in traces if trace_id_of(t) not in completed_ids]
        print(f"Resuming: {len(results)} already done, {len(traces)} remaining")
    else:
        results = []

    total_traces = len(results) + len(traces)

    for record in traces:
        trace_id = trace_id_of(record)
        trace_text = record['trace']['trajectory'] if isinstance(record['trace'], dict) else record['trace']

        if len(trace_text) + len(judge.examples) > 1048570:
            trace_text = trace_text[:1048570 - len(judge.examples)]

        try:
            response = judge.judge_trace(trace_id, trace_text)
            results.append(response)
            print(f"  {len(results)}/{total_traces}")

            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)

            if len(results) % 10 == 0:
                with open(f'saved_results/checkpoints/{cfg.name}_backup_{len(results)}.pkl', 'wb') as f:
                    pickle.dump(results, f)

        except Exception as e:
            print(f"Error on trace {len(results)}: {str(e)}")
            with open(checkpoint_path, 'wb') as f:
                pickle.dump(results, f)

    print(f"Progress: {len(results)}/{total_traces} traces completed")

    rows = [{"trace_id": r.trace_id, "model": r.model_id,
             "name": cfg.name,
             "tokens_in": r.tokens_in, "tokens_out": r.tokens_out,
             "latency_s": r.latency_s, "cost_usd": r.cost_usd,
             **r.annotations} for r in results]
    all_predictions.extend(rows)

    # Ground truth keyed by trace_id — safe for resume (no ordering assumption)
    all_raw = load_dataset(cfg)
    if all_raw and "round" in all_raw[0]:
        all_raw = [t for t in all_raw if t.get("round") == "Round 3"]
    gt_lookup = build_gt_lookup(all_raw, official_modes)

    # Align predictions and ground truth in results order
    ground_truth = {code: [gt_lookup.get(r.trace_id, {}).get(code, 0) for r in results]
                    for code in official_modes}
    failure_mode_results = {mode: [r.annotations[mode] for r in results] for mode in official_modes}

    y_true_flat = []
    y_pred_flat = []
    for mode in official_modes:
        y_true_flat.extend(ground_truth[mode])
        y_pred_flat.extend(failure_mode_results[mode])

    print(f"Accuracy: {accuracy_score(y_true_flat, y_pred_flat):.2f}  |  "
          f"Precision: {precision_score(y_true_flat, y_pred_flat, zero_division=0):.2f}  |  "
          f"Recall: {recall_score(y_true_flat, y_pred_flat, zero_division=0):.2f}  |  "
          f"F1: {f1_score(y_true_flat, y_pred_flat, zero_division=0):.2f}  |  "
          f"Kappa: {cohen_kappa_score(y_true_flat, y_pred_flat):.2f}")

    rows_fm = []
    for mode in official_modes:
        yt = ground_truth[mode]
        yp = failure_mode_results[mode]
        rows_fm.append({
            "name": cfg.name,
            "model": cfg.model,
            "mode": mode,
            "precision": precision_score(yt, yp, zero_division=0),
            "recall":    recall_score(yt, yp, zero_division=0),
            "f1":        f1_score(yt, yp, zero_division=0),
            "support":   sum(yt),
        })
    all_metrics_per_mode.extend(rows_fm)

    yt = np.array(y_true_flat)
    yp = np.array(y_pred_flat)

    f1_lo, f1_hi = bootstrap_ci(yt, yp, lambda a, b: f1_score(a, b, zero_division=0))
    kappa_lo, kappa_hi = bootstrap_ci(yt, yp, cohen_kappa_score)

    print(f"F1: {f1_score(yt, yp, zero_division=0):.3f}  95% CI [{f1_lo:.3f}, {f1_hi:.3f}]")
    print(f"Kappa: {cohen_kappa_score(yt, yp):.3f}  95% CI [{kappa_lo:.3f}, {kappa_hi:.3f}]")

    summary.append({
        "name": cfg.name,
        "model": cfg.model,
        "accuracy":  accuracy_score(y_true_flat, y_pred_flat),
        "precision": precision_score(y_true_flat, y_pred_flat, zero_division=0),
        "recall":    recall_score(y_true_flat, y_pred_flat, zero_division=0),
        "f1":        f1_score(y_true_flat, y_pred_flat, zero_division=0),
        "kappa":     cohen_kappa_score(y_true_flat, y_pred_flat),
        "f1_ci_lo": f1_lo, "f1_ci_hi": f1_hi,
        "kappa_ci_lo": kappa_lo, "kappa_ci_hi": kappa_hi,
        "total_cost_usd":  sum(r.cost_usd for r in results),
        "mean_cost_usd":   sum(r.cost_usd for r in results) / len(results),
        "total_latency_s": sum(r.latency_s for r in results),
        "mean_latency_s":  sum(r.latency_s for r in results) / len(results),
    })

pd.DataFrame(summary).to_csv("saved_results/summary.csv", index=False)
pd.DataFrame(all_metrics_per_mode).to_csv("saved_results/metrics_per_mode.csv", index=False)
pd.DataFrame(all_predictions).to_csv("saved_results/predictions.csv", index=False)

### 3. Results

Summary table (one row per model) and per-mode F1 breakdown.

In [ ]:
summary_df = pd.DataFrame(summary)
display_cols = ['name', 'model', 'f1', 'f1_ci_lo', 'f1_ci_hi', 'kappa', 'kappa_ci_lo', 'kappa_ci_hi',
                'accuracy', 'precision', 'recall', 'mean_cost_usd', 'mean_latency_s', 'total_cost_usd']
summary_df[display_cols].round(3)

In [ ]:
mode_df = pd.DataFrame(all_metrics_per_mode)
pivot = mode_df.pivot_table(index='name', columns='mode', values='f1').round(3)
pivot.insert(0, 'macro_f1', summary_df.set_index('name')['f1'].round(3))
pivot

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

s = summary_df.copy()
fig, axes = plt.subplots(1, 2, figsize=(15, max(4, len(s) * 0.6 + 2)))

# Left: overall F1 with 95% CI error bars
xerr = [s['f1'] - s['f1_ci_lo'], s['f1_ci_hi'] - s['f1']]
bars = axes[0].barh(s['name'], s['f1'], xerr=xerr, capsize=4, color='steelblue', alpha=0.8)
axes[0].set_xlim(0, 1)
axes[0].set_xlabel('Macro F1')
axes[0].set_title('Overall F1 by model (95% CI)')
axes[0].xaxis.set_major_formatter(mticker.FormatStrFormatter('%.2f'))
for bar, val in zip(bars, s['f1']):
    axes[0].text(bar.get_width() + 0.01, bar.get_y() + bar.get_height() / 2,
                 f'{val:.3f}', va='center', fontsize=9)

# Right: per-mode F1 heatmap-style line plot
for name in pivot.index:
    axes[1].plot(official_modes, pivot.loc[name, official_modes].values, marker='o', label=name)
axes[1].set_ylim(0, 1)
axes[1].set_xlabel('Failure mode')
axes[1].set_ylabel('F1')
axes[1].set_title('Per-mode F1 by model')
axes[1].tick_params(axis='x', rotation=45)
axes[1].legend(fontsize=8)
axes[1].grid(axis='y', linestyle='--', alpha=0.5)

plt.tight_layout()
os.makedirs('saved_results', exist_ok=True)
plt.savefig('saved_results/baseline_figure.png', dpi=150, bbox_inches='tight')
plt.show()